# Process FewRel Intermediates into Pipeline Pickles

Run `create_FewRel.ipynb` first to create the FewRel relation split and full-object episode JSON. This notebook converts those intermediates into the same pickle shapes as:

- `data/fs_tacred_train_non_split_original_samples.pkl`
- `data/fs_tacred_final_step_dev_episodes_1shots.pkl`
- `data/fs_tacred_final_step_dev_episodes_shots_details.pkl`


In [1]:
from pathlib import Path
import json
import pickle
from collections import defaultdict, Counter

DATA_DIR = Path('../data')

N_WAY = 5
K_SHOT = 1
NO_RELATION_LABEL = 'no_relation'
NUM_DEV_EPISODES = 9000
EPISODE_SEED = 160292

TRAIN_RELATION_JSON_PATH = DATA_DIR / 'fewrel_train_relation_samples.json'
DEV_EPISODES_JSON_PATH = DATA_DIR / f'fewrel_dev_{N_WAY}_way_{K_SHOT}_shots_{NUM_DEV_EPISODES}episodes_seed{EPISODE_SEED}.json'
RELATION_SPLIT_PATH = DATA_DIR / 'fewrel_relation_split.json'

TRAIN_PICKLE_PATH = DATA_DIR / 'fs_fewrel_train_non_split_original_samples.pkl'
DEV_EPISODES_PICKLE_PATH = DATA_DIR / 'fs_fewrel_final_step_dev_episodes_1shots.pkl'
DEV_DETAILS_PICKLE_PATH = DATA_DIR / 'fs_fewrel_final_step_dev_episodes_shots_details.pkl'
METADATA_PATH = DATA_DIR / 'fs_fewrel_final_step_dev_metadata.json'


In [2]:
def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(data, path, indent=2):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=indent, ensure_ascii=False)

def save_pickle(data, path, protocol=pickle.HIGHEST_PROTOCOL):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(data, f, protocol=protocol)

def has_valid_entity_spans(sample):
    tokens = sample.get('token', [])
    n_tokens = len(tokens)
    span_fields = ['subj_start', 'subj_end', 'obj_start', 'obj_end']
    if n_tokens == 0:
        return False
    if any(field not in sample for field in span_fields):
        return False
    if not all(isinstance(sample[field], int) for field in span_fields):
        return False
    if sample['subj_start'] > sample['subj_end'] or sample['obj_start'] > sample['obj_end']:
        return False
    return (
        0 <= sample['subj_start'] < n_tokens
        and 0 <= sample['subj_end'] < n_tokens
        and 0 <= sample['obj_start'] < n_tokens
        and 0 <= sample['obj_end'] < n_tokens
    )

def extract_shot(sample):
    # Match the TACRED details/train sample schema exactly.
    return {
        'id': sample['id'],
        'relation': sample['relation'],
        'token': list(sample['token']),
        'subj_start': sample['subj_start'],
        'subj_end': sample['subj_end'],
        'obj_start': sample['obj_start'],
        'obj_end': sample['obj_end'],
        'subj_type': sample.get('subj_type'),
        'obj_type': sample.get('obj_type'),
    }

def normalize_query_for_episode(query):
    # Artificial no_relation examples may reuse a raw FewRel id that can also appear as a positive query.
    # Give the relabeled negative view its own id before indexing core_data['queries'].
    query = dict(query)
    if query.get('relation') == NO_RELATION_LABEL:
        query.setdefault('original_id', query['id'])
        query.setdefault('original_relation', query.get('relation'))
        if not str(query['id']).endswith('__no_relation'):
            query['id'] = f"{query['id']}__no_relation"
    return query


In [3]:
train_by_relation = read_json(TRAIN_RELATION_JSON_PATH)
dev_episodes_full = read_json(DEV_EPISODES_JSON_PATH)
relation_split = read_json(RELATION_SPLIT_PATH) if RELATION_SPLIT_PATH.exists() else {}

print('train relations:', len(train_by_relation))
print('train examples:', sum(len(v) for v in train_by_relation.values()))
print('dev episodes:', len(dev_episodes_full))


train relations: 54
train examples: 37780
dev episodes: 9000


In [4]:
train_samples = defaultdict(list)
seen_train_ids = set()
dropped_invalid_train_samples = 0

for relation, samples in train_by_relation.items():
    for sample in samples:
        if not has_valid_entity_spans(sample):
            dropped_invalid_train_samples += 1
            continue
        item = extract_shot(sample)
        assert item['relation'] == relation
        assert item['id'] not in seen_train_ids
        seen_train_ids.add(item['id'])
        train_samples[relation].append(item)

print('train pickle type:', type(train_samples))
print('relations:', len(train_samples))
print('examples:', sum(len(v) for v in train_samples.values()))
print('dropped invalid-span train samples:', dropped_invalid_train_samples)
print('min/max per relation:', min(len(v) for v in train_samples.values()), max(len(v) for v in train_samples.values()))


train pickle type: <class 'collections.defaultdict'>
relations: 54
examples: 37780
dropped invalid-span train samples: 0
min/max per relation: 697 700


In [5]:
core_data = {
    'shots': {},
    'queries': {},
    'umbc_shots': {},
}
new_episodes = []
dropped_invalid_episode_count = 0

for episode in dev_episodes_full:
    normalized_queries = [normalize_query_for_episode(query) for query in episode['meta_test']]
    if any(not has_valid_entity_spans(shot) for way in episode['meta_train'] for shot in way):
        dropped_invalid_episode_count += 1
        continue
    if any(not has_valid_entity_spans(query) for query in normalized_queries):
        dropped_invalid_episode_count += 1
        continue

    new_episode = {'meta_train': [], 'meta_test': []}

    for way in episode['meta_train']:
        new_way = []
        for shot_idx, shot in enumerate(way):
            shot_id = shot['id']
            item = extract_shot(shot)
            target_store = core_data['shots'] if shot_idx == 0 else core_data['umbc_shots']
            if shot_id in target_store:
                assert target_store[shot_id] == item
            else:
                target_store[shot_id] = item
            new_way.append(shot_id)
        new_episode['meta_train'].append(new_way)

    for query in normalized_queries:
        query_id = query['id']
        item = extract_shot(query)
        if query_id in core_data['queries']:
            assert core_data['queries'][query_id] == item
        else:
            core_data['queries'][query_id] = item
        new_episode['meta_test'].append(query_id)

    new_episodes.append(new_episode)

print('episodes:', len(new_episodes))
print('unique shots:', len(core_data['shots']))
print('unique queries:', len(core_data['queries']))
print('unique umbc_shots:', len(core_data['umbc_shots']))
print('dropped invalid-span episodes:', dropped_invalid_episode_count)
no_relation_query_count = sum(
    sample['relation'] == NO_RELATION_LABEL
    for sample in core_data['queries'].values()
)
episode_no_relation_count = sum(
    any(core_data['queries'][query_id]['relation'] == NO_RELATION_LABEL for query_id in episode['meta_test'])
    for episode in new_episodes
)
print('unique no_relation queries:', no_relation_query_count, '/', len(core_data['queries']), f'({no_relation_query_count / len(core_data["queries"]):.2%})')
print('episodes with no_relation:', episode_no_relation_count, '/', len(new_episodes), f'({episode_no_relation_count / len(new_episodes):.2%})')
print('first episode:', new_episodes[0])


episodes: 9000
unique shots: 6991
unique queries: 8227
unique umbc_shots: 0
dropped invalid-span episodes: 0
unique no_relation queries: 7548 / 8227 (91.75%)
episodes with no_relation: 8277 / 9000 (91.97%)
first episode: {'meta_train': [['P86_112'], ['P102_100'], ['P355_135'], ['P407_392'], ['P1346_542']], 'meta_test': ['P17_554__no_relation']}


In [6]:
# Format checks against the existing fs_tacred_* conventions.
dev_support_relations = set()
dev_query_relations = set()

for episode in new_episodes:
    assert len(episode['meta_train']) == N_WAY
    assert all(len(way) == K_SHOT for way in episode['meta_train'])
    assert len(episode['meta_test']) == 1

    support_ids = {shot_id for way in episode['meta_train'] for shot_id in way}
    query_ids = set(episode['meta_test'])
    assert support_ids.isdisjoint(query_ids)

    way_relations = {core_data['shots'][way[0]]['relation'] for way in episode['meta_train']}
    for query_id in episode['meta_test']:
        query_relation = core_data['queries'][query_id]['relation']
        assert query_relation == NO_RELATION_LABEL or query_relation in way_relations

for sample in core_data['shots'].values():
    dev_support_relations.add(sample['relation'])
for sample in core_data['queries'].values():
    dev_query_relations.add(sample['relation'])

train_relations = set(train_samples)
assert train_relations.isdisjoint(dev_support_relations)
assert train_relations.isdisjoint(dev_query_relations)

print('dev support relations:', len(dev_support_relations))
print('dev query relations:', len(dev_query_relations))
print('train/dev relation overlap:', len(train_relations & (dev_support_relations | dev_query_relations)))
print('episodes with no_relation:', episode_no_relation_count, '/', len(new_episodes), f'({episode_no_relation_count / len(new_episodes):.2%})')
print('sanity checks passed')


dev support relations: 10
dev query relations: 11
train/dev relation overlap: 0
episodes with no_relation: 8277 / 9000 (91.97%)
sanity checks passed


In [7]:
metadata = {
    'dataset': 'fewrel',
    'train_pickle_path': str(TRAIN_PICKLE_PATH),
    'dev_episodes_pickle_path': str(DEV_EPISODES_PICKLE_PATH),
    'dev_details_pickle_path': str(DEV_DETAILS_PICKLE_PATH),
    'n_way': N_WAY,
    'k_shot': K_SHOT,
    'num_dev_episodes': len(new_episodes),
    'episode_seed': EPISODE_SEED,
    'train_relations': sorted(train_samples),
    'dev_relations': sorted(dev_support_relations | dev_query_relations),
    'relation_split': relation_split,
    'num_train_examples': sum(len(v) for v in train_samples.values()),
    'num_unique_dev_shots': len(core_data['shots']),
    'num_unique_dev_queries': len(core_data['queries']),
    'num_unique_no_relation_dev_queries': no_relation_query_count,
    'num_episodes_with_no_relation': episode_no_relation_count,
    'num_dropped_invalid_train_samples': dropped_invalid_train_samples,
    'num_dropped_invalid_dev_episodes': dropped_invalid_episode_count,
    'num_unique_dev_umbc_shots': len(core_data['umbc_shots']),
}

metadata


{'dataset': 'fewrel',
 'train_pickle_path': '../data/fs_fewrel_train_non_split_original_samples.pkl',
 'dev_episodes_pickle_path': '../data/fs_fewrel_final_step_dev_episodes_1shots.pkl',
 'dev_details_pickle_path': '../data/fs_fewrel_final_step_dev_episodes_shots_details.pkl',
 'n_way': 5,
 'k_shot': 1,
 'num_dev_episodes': 9000,
 'episode_seed': 160292,
 'train_relations': ['applies to jurisdiction',
  'architect',
  'characters',
  'contains administrative territorial entity',
  'country',
  'country of citizenship',
  'country of origin',
  'developer',
  'director',
  'father',
  'field of work',
  'followed by',
  'genre',
  'has part',
  'head of government',
  'headquarters location',
  'heritage designation',
  'instance of',
  'instrument',
  'league',
  'located in the administrative territorial entity',
  'located on terrain feature',
  'location',
  'manufacturer',
  'military branch',
  'mountain range',
  'mouth of the watercourse',
  'movement',
  'nominated for',
  'not

In [8]:
# Run this cell when you are ready to create the final pipeline pickle files.
save_pickle(train_samples, TRAIN_PICKLE_PATH)
save_pickle(new_episodes, DEV_EPISODES_PICKLE_PATH)
save_pickle(core_data, DEV_DETAILS_PICKLE_PATH)
save_json(metadata, METADATA_PATH)

print('wrote:', TRAIN_PICKLE_PATH)
print('wrote:', DEV_EPISODES_PICKLE_PATH)
print('wrote:', DEV_DETAILS_PICKLE_PATH)
print('wrote:', METADATA_PATH)


wrote: ../data/fs_fewrel_train_non_split_original_samples.pkl
wrote: ../data/fs_fewrel_final_step_dev_episodes_1shots.pkl
wrote: ../data/fs_fewrel_final_step_dev_episodes_shots_details.pkl
wrote: ../data/fs_fewrel_final_step_dev_metadata.json
